In [0]:
%sql
-- Rated points lookup
CREATE OR REPLACE TABLE finance_project.rated_points (
  grade string, points int
);

In [0]:
%sql
INSERT INTO finance_project.rated_points VALUES
('Unacceptable', 0), ('Very Bad', 100), ('Bad', 250), 
('Good', 500), ('Very Good', 650), ('Excellent', 800);

In [0]:
%sql
CREATE OR REPLACE TABLE finance_project.grade_points (
  grade string, points int
);


In [0]:
%sql
INSERT INTO finance_project.grade_points VALUES
('Unacceptable', 750), ('Very Bad', 1000), ('Bad', 1500), 
('Good', 2000), ('Very Good', 2500);

In [0]:
dbutils.fs.ls("/Volumes/finance_domain_project/default/raw_data/BadData")


In [0]:
bad_customer_data_final_df = spark.read \
.format("csv") \
.option("header",True) \
.option("inferSchema",True) \
.load("/Volumes/finance_domain_project/default/raw_data/BadData/bad_customer_data_final")

In [0]:
bad_customer_data_final_df.createOrReplaceTempView("bad_data_customer")

In [0]:
%sql
SELECT member_id, COUNT(*) as duplicate_count
FROM bad_data_customer 
GROUP BY member_id 
HAVING COUNT(*) > 1
ORDER BY duplicate_count DESC

In [0]:
total_rows_df = spark.sql("SELECT COUNT(*) as total_rows FROM bad_data_customer")
display(total_rows_df)

In [0]:
%sql
SELECT 
  COUNT(*) as total_rows,
  COUNT(DISTINCT member_id) as unique_member_ids
FROM bad_data_customer;

In [0]:
dbutils.fs.ls("/Volumes/finance_domain_project/default/raw_data/BadData/bad_customer_data_final/")

In [0]:
%sql
SHOW TABLES IN finance_project;

In [0]:
ph_df = spark.sql("""
  SELECT 
    c.member_id,
    CASE 
      WHEN p.last_payment_amount < (c.monthly_installment * 0.5) THEN 100
      WHEN p.last_payment_amount >= (c.monthly_installment * 0.5) 
           AND p.last_payment_amount < c.monthly_installment THEN 250
      WHEN p.last_payment_amount = c.monthly_installment THEN 500
      WHEN p.last_payment_amount > c.monthly_installment 
           AND p.last_payment_amount <= (c.monthly_installment * 1.50) THEN 650
      WHEN p.last_payment_amount > (c.monthly_installment * 1.50) THEN 800
      ELSE 0
    END as last_payment_points,
    
    CASE 
      WHEN p.total_payment_received >= (c.funded_amount * 0.50) THEN 650
      WHEN p.total_payment_received < (c.funded_amount * 0.50) 
           AND p.total_payment_received > 0 THEN 500
      WHEN p.total_payment_received = 0 OR p.total_payment_received IS NULL THEN 0
      ELSE 0
    END as total_payment_points
    
  FROM finance_project.loans_repayments_external p
  INNER JOIN finance_project.loans_external c ON c.loan_id = p.loan_id
  WHERE c.member_id NOT IN (SELECT member_id FROM bad_data_customer)
""")

In [0]:
ph_df.show(10)
ph_df.groupBy("last_payment_points").count().orderBy("count", ascending=False).show()

In [0]:
ph_df.createOrReplaceTempView("ph_pts")

In [0]:
spark.sql("select * from ph_pts").show()

In [0]:
ldh_ph_df = spark.sql("""
  SELECT 
    p.*,
    CASE 
      WHEN d.delinq_2yrs = 0 THEN 800
      WHEN d.delinq_2yrs BETWEEN 1 AND 2 THEN 250
      WHEN d.delinq_2yrs BETWEEN 3 AND 5 THEN 100
      WHEN d.delinq_2yrs > 5 OR d.delinq_2yrs IS NULL THEN 0
    END AS delinq_pts,
    
    CASE 
      WHEN l.pub_rec = 0 THEN 800
      WHEN l.pub_rec BETWEEN 1 AND 2 THEN 250
      WHEN l.pub_rec BETWEEN 3 AND 5 THEN 100
      WHEN l.pub_rec > 5 OR l.pub_rec IS NULL THEN 100
    END AS public_records_pts,
    
    CASE 
      WHEN l.pub_rec_bankruptcies = 0 THEN 800
      WHEN l.pub_rec_bankruptcies BETWEEN 1 AND 2 THEN 250
      WHEN l.pub_rec_bankruptcies BETWEEN 3 AND 5 THEN 100
      WHEN l.pub_rec_bankruptcies > 5 OR l.pub_rec_bankruptcies IS NULL THEN 100
    END AS public_bankruptcies_pts,
    
    CASE 
      WHEN l.inq_last_6mths = 0 THEN 800
      WHEN l.inq_last_6mths BETWEEN 1 AND 2 THEN 250
      WHEN l.inq_last_6mths BETWEEN 3 AND 5 THEN 100
      WHEN l.inq_last_6mths > 5 OR l.inq_last_6mths IS NULL THEN 0
    END AS enq_pts
    
  FROM finance_project.loans_defaulters_detail_rec_enq_new l
  INNER JOIN finance_project.loans_defaulters_delinq_new d 
    ON d.member_id = l.member_id
  INNER JOIN ph_pts p 
    ON p.member_id = l.member_id 
  WHERE l.member_id NOT IN (SELECT member_id FROM bad_data_customer)
""")

In [0]:
# Check structure
ldh_ph_df.printSchema()

# See rating distributions
ldh_ph_df.groupBy("delinq_pts").count().show()
ldh_ph_df.groupBy("public_records_pts").count().show()

# Sample data
ldh_ph_df.show(5)

In [0]:
ldh_ph_df.createOrReplaceTempView("ldh_ph_pts")


In [0]:
spark.sql("select * from ldh_ph_pts").show()

In [0]:
fh_ldh_ph_df = spark.sql("""
  SELECT 
    ldef.*,
    CASE 
      WHEN LOWER(l.loan_status) LIKE '%fully paid%' THEN 800
      WHEN LOWER(l.loan_status) LIKE '%current%' THEN 500
      WHEN LOWER(l.loan_status) LIKE '%in grace period%' THEN 250
      WHEN LOWER(l.loan_status) LIKE '%late (16-30 days)%' 
           OR LOWER(l.loan_status) LIKE '%late (31-120 days)%' THEN 100
      WHEN LOWER(l.loan_status) LIKE '%charged off%' THEN 0
      ELSE 0
    END AS loan_status_pts,
    
    CASE 
      WHEN LOWER(a.home_ownership) LIKE '%own' THEN 800
      WHEN LOWER(a.home_ownership) LIKE '%rent' THEN 500
      WHEN LOWER(a.home_ownership) LIKE '%mortgage' THEN 250
      WHEN LOWER(a.home_ownership) LIKE '%any' OR LOWER(a.home_ownership) IS NULL THEN 100
    END AS home_pts,
    
    CASE 
      WHEN l.funded_amount <= (a.tot_hi_cred_limit * 0.10) THEN 800
      WHEN l.funded_amount > (a.tot_hi_cred_limit * 0.10) 
           AND l.funded_amount <= (a.tot_hi_cred_limit * 0.20) THEN 650
      WHEN l.funded_amount > (a.tot_hi_cred_limit * 0.20) 
           AND l.funded_amount <= (a.tot_hi_cred_limit * 0.30) THEN 500
      WHEN l.funded_amount > (a.tot_hi_cred_limit * 0.30) 
           AND l.funded_amount <= (a.tot_hi_cred_limit * 0.50) THEN 250
      WHEN l.funded_amount > (a.tot_hi_cred_limit * 0.50) 
           AND l.funded_amount <= (a.tot_hi_cred_limit * 0.70) THEN 100
      WHEN l.funded_amount > (a.tot_hi_cred_limit * 0.70) THEN 0
      ELSE 0
    END AS credit_limit_pts,
    
    CASE 
      WHEN a.grade = 'A' AND a.sub_grade = 'A1' THEN 2500
      WHEN a.grade = 'A' AND a.sub_grade = 'A2' THEN 2500
      WHEN a.grade = 'A' AND a.sub_grade = 'A3' THEN 2500
      WHEN a.grade = 'A' AND a.sub_grade = 'A4' THEN 2500
      WHEN a.grade = 'A' AND a.sub_grade = 'A5' THEN 2000

      WHEN a.grade = 'B' AND a.sub_grade = 'B1' THEN 2000
      WHEN a.grade = 'B' AND a.sub_grade = 'B2' THEN 2000
      WHEN a.grade = 'B' AND a.sub_grade = 'B3' THEN 2000
      WHEN a.grade = 'B' AND a.sub_grade = 'B4' THEN 2000
      WHEN a.grade = 'B' AND a.sub_grade = 'B5' THEN 1500

      WHEN a.grade = 'C' AND a.sub_grade = 'C1' THEN 1500
      WHEN a.grade = 'C' AND a.sub_grade = 'C2' THEN 1500
      WHEN a.grade = 'C' AND a.sub_grade = 'C3' THEN 1500
      WHEN a.grade = 'C' AND a.sub_grade = 'C4' THEN 1500
      WHEN a.grade = 'C' AND a.sub_grade = 'C5' THEN 1000

      WHEN a.grade = 'D' AND a.sub_grade = 'D1' THEN 1000
      WHEN a.grade = 'D' AND a.sub_grade = 'D2' THEN 1000
      WHEN a.grade = 'D' AND a.sub_grade = 'D3' THEN 1000
      WHEN a.grade = 'D' AND a.sub_grade = 'D4' THEN 1000
      WHEN a.grade = 'D' AND a.sub_grade = 'D5' THEN 750

      WHEN a.grade = 'E' AND a.sub_grade = 'E1' THEN 750
      WHEN a.grade = 'E' AND a.sub_grade = 'E2' THEN 750
      WHEN a.grade = 'E' AND a.sub_grade = 'E3' THEN 750
      WHEN a.grade = 'E' AND a.sub_grade = 'E4' THEN 750
      WHEN a.grade = 'E' AND a.sub_grade = 'E5' THEN 750

      WHEN a.grade IN ('F', 'G') THEN 750
      ELSE 750
    END AS grade_pts
    
  FROM ldh_ph_pts ldef
  INNER JOIN finance_project.loans_external l 
    ON ldef.member_id = l.member_id
  INNER JOIN finance_project.customers_new a 
    ON a.member_id = ldef.member_id 
  WHERE ldef.member_id NOT IN (SELECT member_id FROM bad_data_customer)
""")

In [0]:
fh_ldh_ph_df.createOrReplaceTempView("fh_ldh_ph_pts")

In [0]:
spark.sql("select * from fh_ldh_ph_pts").show()

In [0]:
loan_score = spark.sql("""
  SELECT 
    member_id,
    (last_payment_points + total_payment_points) * 0.25 AS payment_history_pts,
    (delinq_pts + public_records_pts + public_bankruptcies_pts + enq_pts) * 0.45 AS defaulters_history_pts,
    (loan_status_pts + home_pts + credit_limit_pts + grade_pts) * 0.30 AS financial_health_pts
  FROM fh_ldh_ph_pts
""")

In [0]:
final_loan_score = loan_score.withColumn('loan_score', 
    loan_score.payment_history_pts + loan_score.defaulters_history_pts + loan_score.financial_health_pts
)

In [0]:
final_loan_score.createOrReplaceTempView("loan_score_eval")

In [0]:
final_loan_score.printSchema()


In [0]:
spark.sql("SELECT * FROM loan_score_eval").show(5)

In [0]:
spark.sql("SELECT member_id, delinq_pts, loan_status_pts FROM fh_ldh_ph_pts LIMIT 5").show()

In [0]:
# Use dbutils widgets to set the values
dbutils.widgets.text("very_good_grade_pts", "2500")
dbutils.widgets.text("good_grade_pts", "1500")
dbutils.widgets.text("bad_grade_pts", "1000")
dbutils.widgets.text("very_bad_grade_pts", "500")
dbutils.widgets.text("unacceptable_grade_pts", "0")

In [0]:
loan_score_final = spark.sql("""
SELECT ls.*,
CASE
WHEN loan_score > 2500 THEN 'A'
WHEN loan_score <= 2500 AND loan_score > 1500 THEN 'B'
WHEN loan_score <= 1500 AND loan_score > 1000 THEN 'C'
WHEN loan_score <= 1000 AND loan_score > 500 THEN 'D'
WHEN loan_score <= 500 AND loan_score > 0 THEN 'E'
WHEN loan_score <= 0 THEN 'F'
END AS loan_final_grade
FROM loan_score_eval ls
""")

In [0]:
loan_score_final.createOrReplaceTempView("loan_final_table")

In [0]:
spark.sql("select * from loan_final_table where loan_final_grade in ('A')").show(5)


In [0]:
spark.sql("select count(*) from loan_final_table").show()

In [0]:
dbutils.fs.mkdirs("/Volumes/finance_domain_project/default/raw_data/Processed")


In [0]:
dbutils.fs.ls("/Volumes/finance_domain_project/default/raw_data")

In [0]:
loan_score_final.write \
    .format("parquet") \
    .mode("overwrite") \
    .option("path", "/Volumes/finance_domain_project/default/raw_data/Processed/loan_score") \
    .save()